# Notebook 3 — MongoDB Atlas: NoSQL Design, CRUD & Query Optimisation
## NorthStar Urban Mobility and Logistics — Case Study
---
**What this notebook does:**
- Connects to MongoDB Atlas using PyMongo
- Designs 3 document collections that solve NorthStar's data fragmentation
- Performs all CRUD operations with real NorthStar data
- Creates indexes and demonstrates query optimisation with explain plans


## Section 1 — Install PyMongo & Connect to Atlas

In [1]:
# Install pymongo with the SRV extra for Atlas connections
!pip install "pymongo[srv]" --quiet
print("PyMongo installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.2 MB/s eta 0:00:00
PyMongo installed.


In [2]:
import pymongo
import pandas as pd
import json
from datetime import datetime
from pprint import pprint

print(f"PyMongo version: {pymongo.__version__}")

PyMongo version: 4.16.0


In [3]:
# MongoDB Atlas Connection
from urllib.parse import quote_plus

username = "northstar_user"
password = "northstar_user@123"
host     = "cluster0.dr4bf7p.mongodb.net"

# URL-encode the password so the @ in it doesn't break the connection string
password_encoded = quote_plus(password)

MONGO_URI = (f"mongodb+srv://{username}:{password_encoded}@{host}"f"/?retryWrites=true&w=majority&appName=Cluster0")

# Connect
client = pymongo.MongoClient(MONGO_URI)

# Test connection — ping the server
client.admin.command("ping")
print("Successfully connected to MongoDB Atlas!")
print(f"Server info: {client.server_info()['version']}")

Successfully connected to MongoDB Atlas!
Server info: 8.0.20


In [4]:
# Create (or connect to) the NorthStar database
db = client["northstar_db"]
print(f"Using database: northstar_db")
print(f"Existing collections: {db.list_collection_names()}")

Using database: northstar_db
Existing collections: ['app_sessions', 'delivery_events', 'customer_cases']


## Section 2 — Why MongoDB? NoSQL Design Justification



| File | Why MongoDB? |
|------|-------------|
| `app_events` | Events have variable structure, belong to sessions, and grow over time. Grouping by session makes querying far faster than SQL JOINs across millions of rows |
| `complaints` | Each complaint has evolving status, variable escalation history, and compensation records. Embedding per customer avoids repeated JOINs |
| `incidents` | Each delivery can have 0 to many incidents of different types. Embedding as an array inside the delivery document keeps the full event history together |

The **3 collections** we create:
1. `customer_cases` — one document per customer containing all their complaints as a nested array
2. `delivery_events` — one document per delivery containing all its incidents as a nested array
3. `app_sessions` — one document per app session containing all events in that session


## Section 3 — Load Clean Data

In [ ]:
# Load cleaned CSV files directly from GitHub for MongoDB processing
clean_base_url = "https://raw.githubusercontent.com/m2007mahesh-byte/Northstar-case-study/main/Cleaned%20Dataset/"

customers   = pd.read_csv(clean_base_url + "customers_clean.csv")
complaints  = pd.read_csv(clean_base_url + "complaints_clean.csv")
deliveries  = pd.read_csv(clean_base_url + "deliveries_clean.csv")
incidents   = pd.read_csv(clean_base_url + "incidents_clean.csv")
app_events  = pd.read_csv(clean_base_url + "app_events_clean.csv")
orders      = pd.read_csv(clean_base_url + "orders_clean.csv")

print(f"customers  : {len(customers)} rows")
print(f"complaints : {len(complaints)} rows")
print(f"deliveries : {len(deliveries)} rows")
print(f"incidents  : {len(incidents)} rows")
print(f"app_events : {len(app_events)} rows")
print(f"orders     : {len(orders)} rows")

customers  : 650 rows
complaints : 320 rows
deliveries : 950 rows
incidents  : 280 rows
app_events : 640 rows


## Section 4 — Collection 1: `customer_cases`
### Design: One document per customer with all complaints embedded as an array

In [7]:
# Drop collection if it already exists
db.customer_cases.drop()
print("Collection cleared.")

Collection cleared.


In [8]:
# Build customer_cases documents
# Each document = 1 customer + their profile + all their complaints as a nested list

customer_docs = []

for _, cust in customers.iterrows():
    # Find all complaints for this customer
    cust_complaints = complaints[complaints["customer_id"] == cust["customer_id"]]

    # Build the nested complaint array
    complaint_list = []
    for _, comp in cust_complaints.iterrows():
        complaint_list.append({
            "complaint_id"       : comp["complaint_id"],
            "order_id"           : comp["order_id"],
            "complaint_type"     : comp["complaint_type"],
            "channel"            : comp["channel"],
            "severity"           : comp["severity"],
            "created_at"         : comp["created_at"],
            "status"             : comp["status"],
            "resolution_days"    : int(comp["resolution_days"]),
            "compensation_amount": float(comp["compensation_amount"])
        })

    # Main customer document
    doc = {
        "customer_id"          : cust["customer_id"],
        "age"                  : int(cust["age"]),
        "home_zone"            : cust["home_zone"],
        "customer_type"        : cust["customer_type"],
        "signup_date"          : cust["signup_date"],
        "loyalty_score"        : float(cust["loyalty_score"]),
        "app_engagement_score" : float(cust["app_engagement_score"]),
        "preferred_channel"    : cust["preferred_channel"],
        "account_status"       : cust["account_status"],
        "repeat_complainer"    : int(cust["repeat_complainer"]) if "repeat_complainer" in cust else 0,
        "total_complaints"     : len(complaint_list),
        "complaints"           : complaint_list   # nested array
    }
    customer_docs.append(doc)

print(f"Built {len(customer_docs)} customer documents.")
print()
print("Example document structure (first customer with complaints):")
for doc in customer_docs:
    if doc["total_complaints"] > 0:
        pprint(doc, depth=3)
        break

Built 650 customer documents.

Example document structure (first customer with complaints):
{'account_status': 'Active',
 'age': 26,
 'app_engagement_score': 69.2,
 'complaints': [{'channel': 'App',
                 'compensation_amount': 43.9,
                 'complaint_id': 'CP0096',
                 'complaint_type': 'AppIssue',
                 'created_at': '2024-05-12 21:32:00',
                 'order_id': 'O00007',
                 'resolution_days': 22,
                 'severity': 'High',
                 'status': 'Resolved'},
                {'channel': 'Phone',
                 'compensation_amount': 0.0,
                 'complaint_id': 'CP0146',
                 'complaint_type': 'Delay',
                 'created_at': '2025-09-01 20:17:00',
                 'order_id': 'O00666',
                 'resolution_days': 4,
                 'severity': 'Medium',
                 'status': 'Resolved'}],
 'customer_id': 'C0001',
 'customer_type': 'SME',
 'home_zone': 'North',
 

In [9]:
# INSERT — bulk insert all customer documents
result = db.customer_cases.insert_many(customer_docs)
print(f"Inserted {len(result.inserted_ids)} documents into customer_cases")

Inserted 650 documents into customer_cases


In [10]:
# READ — verify insertion and sample a document
total = db.customer_cases.count_documents({})
print(f"Total documents in customer_cases: {total}")
print()
print("Sample document (customer with at least 1 complaint):")
sample = db.customer_cases.find_one({"total_complaints": {"$gt": 0}})
pprint(sample, depth=4)

Total documents in customer_cases: 650

Sample document (customer with at least 1 complaint):
{'_id': ObjectId('69c3af59660f747ca6b26741'),
 'account_status': 'Active',
 'age': 26,
 'app_engagement_score': 69.2,
 'complaints': [{'channel': 'App',
                 'compensation_amount': 43.9,
                 'complaint_id': 'CP0096',
                 'complaint_type': 'AppIssue',
                 'created_at': '2024-05-12 21:32:00',
                 'order_id': 'O00007',
                 'resolution_days': 22,
                 'severity': 'High',
                 'status': 'Resolved'},
                {'channel': 'Phone',
                 'compensation_amount': 0.0,
                 'complaint_id': 'CP0146',
                 'complaint_type': 'Delay',
                 'created_at': '2025-09-01 20:17:00',
                 'order_id': 'O00666',
                 'resolution_days': 4,
                 'severity': 'Medium',
                 'status': 'Resolved'}],
 'customer_id': 'C0001',
 

## Section 5 — Collection 2: `delivery_events`
### Design: One document per delivery with all incidents embedded

In [11]:
db.delivery_events.drop()
print("Collection cleared.")

Collection cleared.


In [12]:
# Build delivery_events documents
# Each document = 1 delivery + all its incident records as a nested array
# This keeps the complete event history of each delivery in one place

delivery_docs = []

for _, deliv in deliveries.iterrows():
    # Get incidents for this delivery
    deliv_incidents = incidents[incidents["delivery_id"] == deliv["delivery_id"]]

    incident_list = []
    for _, inc in deliv_incidents.iterrows():
        incident_list.append({
            "incident_id"      : inc["incident_id"],
            "incident_type"    : inc["incident_type"],
            "reported_at"      : inc["reported_at"],
            "severity"         : inc["severity"],
            "resolution_status": inc["resolution_status"],
            "resolved_hours"   : float(inc["resolved_hours"])
        })

    doc = {
        "delivery_id"                 : deliv["delivery_id"],
        "order_id"                    : deliv["order_id"],
        "driver_id"                   : deliv["driver_id"],
        "vehicle_id"                  : deliv["vehicle_id"],
        "hub_id"                      : deliv["hub_id"],
        "dispatch_time"               : deliv["dispatch_time"],
        "delivery_completed_at"       : str(deliv["delivery_completed_at"]),
        "delivery_status"             : deliv["delivery_status"],
        "route_distance_km"           : float(deliv["route_distance_km"]),
        "manual_route_override_count" : int(deliv["manual_route_override_count"]),
        "proof_of_completion_missing" : int(deliv["proof_of_completion_missing"]),
        "customer_rating"             : float(deliv["customer_rating_post_delivery"]),
        "fuel_or_charge_cost"         : float(deliv["fuel_or_charge_cost"]),
        "incident_count"              : len(incident_list),
        "incidents"                   : incident_list   # nested array
    }
    delivery_docs.append(doc)

print(f"Built {len(delivery_docs)} delivery documents.")

# Show a delivery that has incidents
for doc in delivery_docs:
    if doc["incident_count"] > 0:
        print("\nExample document with incidents:")
        pprint(doc, depth=4)
        break

Built 950 delivery documents.

Example document with incidents:
{'customer_rating': 3.07,
 'delivery_completed_at': '2024-06-19 09:05:59.904311',
 'delivery_id': 'DL00001',
 'delivery_status': 'Failed',
 'dispatch_time': '2024-06-18 10:57:00',
 'driver_id': 'D004',
 'fuel_or_charge_cost': 12.05,
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_id': 'I0180',
                'incident_type': 'ProofMissing',
                'reported_at': '2024-06-18 11:38:00',
                'resolution_status': 'Open',
                'resolved_hours': 5.6,
                'severity': 'High'}],
 'manual_route_override_count': 1,
 'order_id': 'O00938',
 'proof_of_completion_missing': 0,
 'route_distance_km': 17.26,
 'vehicle_id': 'V056'}


In [13]:
result = db.delivery_events.insert_many(delivery_docs)
print(f"Inserted {len(result.inserted_ids)} documents into delivery_events")

total = db.delivery_events.count_documents({})
print(f"Total in delivery_events: {total}")

Inserted 950 documents into delivery_events
Total in delivery_events: 950


## Section 6 — Collection 3: `app_sessions`
### Design: One document per session with all events as a nested array

In [14]:
db.app_sessions.drop()
print("Collection cleared.")

Collection cleared.


In [15]:
# Build app_sessions documents
# Group all app events by session_id
# This is much better for querying a user's journey through the app than querying individual flat rows

session_groups = app_events.groupby("session_id")
session_docs   = []

for session_id, group in session_groups:
    events_list = []
    for _, row in group.iterrows():
        events_list.append({
            "event_id"      : row["event_id"],
            "event_type"    : row["event_type"],
            "event_timestamp": row["event_timestamp"],
            "device_type"   : row["device_type"],
            "zone_context"  : row["zone_context"],
            "api_latency_ms": int(row["api_latency_ms"]),
            "success_flag"  : int(row["success_flag"]),
            "order_id"      : row["order_id"] if pd.notna(row["order_id"]) else None
        })

    # Session-level summary
    doc = {
        "session_id"         : session_id,
        "customer_id"        : group["customer_id"].iloc[0],
        "device_type"        : group["device_type"].iloc[0],
        "zone_context"       : group["zone_context"].iloc[0],
        "session_start"      : str(group["event_timestamp"].min()),
        "session_end"        : str(group["event_timestamp"].max()),
        "total_events"       : len(events_list),
        "failed_events"      : int((group["success_flag"] == 0).sum()),
        "avg_latency_ms"     : round(float(group["api_latency_ms"].mean()), 1),
        "max_latency_ms"     : int(group["api_latency_ms"].max()),
        "has_payment_retry"  : bool((group["event_type"] == "payment_retry").any()),
        "has_chat_escalation": bool((group["event_type"] == "chat_escalated").any()),
        "events"             : events_list   # nested array of all events
    }
    session_docs.append(doc)

print(f"Built {len(session_docs)} session documents from {len(app_events)} app events.")

# Show a session with multiple events
multi_event = [d for d in session_docs if d["total_events"] > 1]
if multi_event:
    print("\nExample multi-event session document:")
    pprint(multi_event[0], depth=4)

Built 637 session documents from 640 app events.

Example multi-event session document:
{'avg_latency_ms': 921.5,
 'customer_id': 'C0144',
 'device_type': 'Web',
 'events': [{'api_latency_ms': 440,
             'device_type': 'Web',
             'event_id': 'AE00233',
             'event_timestamp': '2025-09-06 23:35:00',
             'event_type': 'search_route',
             'order_id': 'O00990',
             'success_flag': 1,
             'zone_context': 'South'},
            {'api_latency_ms': 1403,
             'device_type': 'Web',
             'event_id': 'AE00462',
             'event_timestamp': '2024-07-08 17:38:00',
             'event_type': 'track_order',
             'order_id': 'O00871',
             'success_flag': 1,
             'zone_context': 'Airport'}],
 'failed_events': 0,
 'has_chat_escalation': False,
 'has_payment_retry': False,
 'max_latency_ms': 1403,
 'session_end': '2025-09-06 23:35:00',
 'session_id': 'S25207',
 'session_start': '2024-07-08 17:38:00',
 '

In [16]:
result = db.app_sessions.insert_many(session_docs)
print(f"Inserted {len(result.inserted_ids)} documents into app_sessions")

total = db.app_sessions.count_documents({})
print(f"Total in app_sessions: {total}")

Inserted 637 documents into app_sessions
Total in app_sessions: 637


## Section 7 — CRUD Operations
### CREATE — already done in sections above. Now READ, UPDATE, DELETE.

### READ Operations

In [17]:
# READ 1: Find all repeat complainers (customers with 2+ complaints)
print("READ 1 — Repeat Complainers:")
print("-" * 40)

repeat_complainers = db.customer_cases.find(
    {"total_complaints": {"$gte": 2}},
    {"customer_id": 1, "customer_type": 1, "home_zone": 1,
     "total_complaints": 1, "_id": 0}
).sort("total_complaints", -1).limit(10)

for doc in repeat_complainers:
    print(doc)

READ 1 — Repeat Complainers:
----------------------------------------
{'customer_id': 'C0368', 'home_zone': 'North', 'customer_type': 'Consumer', 'total_complaints': 4}
{'customer_id': 'C0372', 'home_zone': 'West', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0242', 'home_zone': 'East', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0191', 'home_zone': 'North', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0282', 'home_zone': 'Riverside', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0142', 'home_zone': 'South', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0545', 'home_zone': 'South', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0421', 'home_zone': 'Central', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0172', 'home_zone': 'North', 'customer_type': 'Consumer', 'total_complaints': 3}
{'customer_id': 'C0110', 'home_zone

In [18]:
# READ 2: Find all failed deliveries from hub H05 with incidents
print("READ 2 — Failed Deliveries from H05 with Incidents:")
print("-" * 50)

failed_h05 = db.delivery_events.find(
    {
        "hub_id"           : "H05",
        "delivery_status"  : "Failed",
        "incident_count"   : {"$gt": 0}
    },
    {
        "delivery_id": 1, "hub_id": 1, "delivery_status": 1,
        "incident_count": 1, "incidents.incident_type": 1,
        "incidents.severity": 1, "_id": 0
    }
)

count = 0
for doc in failed_h05:
    pprint(doc)
    count += 1

print(f"\nTotal failed H05 deliveries with incidents: {count}")

READ 2 — Failed Deliveries from H05 with Incidents:
--------------------------------------------------
{'delivery_id': 'DL00001',
 'delivery_status': 'Failed',
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_type': 'ProofMissing', 'severity': 'High'}]}
{'delivery_id': 'DL00187',
 'delivery_status': 'Failed',
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_type': 'ProofMissing', 'severity': 'High'}]}
{'delivery_id': 'DL00548',
 'delivery_status': 'Failed',
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_type': 'AppSyncError', 'severity': 'Medium'}]}
{'delivery_id': 'DL00574',
 'delivery_status': 'Failed',
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_type': 'AppSyncError', 'severity': 'High'}]}
{'delivery_id': 'DL00694',
 'delivery_status': 'Failed',
 'hub_id': 'H05',
 'incident_count': 1,
 'incidents': [{'incident_type': 'BatteryAlert', 'severity': 'Medium'}]}
{'delivery_id': 'DL00788',
 'delivery_status': 'Faile

In [19]:
# READ 3: Aggregation — count complaints by type across all customers
print("READ 3 — Complaint Type Summary (Aggregation Pipeline):")
print("-" * 50)

pipeline = [
    {"$unwind": "$complaints"},                    # expand the complaints array
    {"$group": {
        "_id"             : "$complaints.complaint_type",
        "total_complaints": {"$sum": 1},
        "avg_compensation": {"$avg": "$complaints.compensation_amount"},
        "high_severity"   : {"$sum": {"$cond": [{"$eq": ["$complaints.severity","High"]}, 1, 0]}}
    }},
    {"$sort": {"total_complaints": -1}}
]

results = db.customer_cases.aggregate(pipeline)
for r in results:
    print(f"  {r['_id']:<22}  count: {r['total_complaints']:>3}  "
          f"avg_comp: £{r['avg_compensation']:.2f}  high_sev: {r['high_severity']}")

READ 3 — Complaint Type Summary (Aggregation Pipeline):
--------------------------------------------------
  Delay                   count: 101  avg_comp: £16.80  high_sev: 18
  MissedPickup            count:  64  avg_comp: £22.24  high_sev: 16
  AppIssue                count:  53  avg_comp: £18.50  high_sev: 13
  DriverBehaviour         count:  51  avg_comp: £19.08  high_sev: 16
  SupportExperience       count:  20  avg_comp: £17.12  high_sev: 3
  Billing                 count:  16  avg_comp: £23.87  high_sev: 4
  Damage                  count:  15  avg_comp: £23.98  high_sev: 7


In [20]:
# READ 4: App sessions with failed payment retries — high business risk
print("READ 4 — Sessions with Failed Payment Retries:")
print("-" * 50)

payment_fail_sessions = db.app_sessions.find(
    {
        "has_payment_retry": True,
        "failed_events"    : {"$gt": 0}
    },
    {
        "session_id": 1, "customer_id": 1, "zone_context": 1,
        "total_events": 1, "failed_events": 1,
        "avg_latency_ms": 1, "_id": 0
    }
).sort("failed_events", -1)

count = 0
for doc in payment_fail_sessions:
    print(doc)
    count += 1

print(f"\nTotal sessions with failed payment retries: {count}")

READ 4 — Sessions with Failed Payment Retries:
--------------------------------------------------
{'session_id': 'S60656', 'customer_id': 'C0303', 'zone_context': 'South', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 637.0}
{'session_id': 'S90136', 'customer_id': 'C0612', 'zone_context': 'Riverside', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 60.0}
{'session_id': 'S89304', 'customer_id': 'C0039', 'zone_context': 'West', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 153.0}
{'session_id': 'S83880', 'customer_id': 'C0381', 'zone_context': 'West', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 93.0}
{'session_id': 'S77602', 'customer_id': 'C0468', 'zone_context': 'Riverside', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 404.0}
{'session_id': 'S77071', 'customer_id': 'C0556', 'zone_context': 'North', 'total_events': 1, 'failed_events': 1, 'avg_latency_ms': 592.0}
{'session_id': 'S74695', 'customer_id': 'C0308', 'zone_context': 'Airp

In [21]:
# READ 5: Aggregation — avg latency by zone from app sessions
print("READ 5 — Average API Latency by Zone:")
print("-" * 45)

latency_pipeline = [
    {"$group": {
        "_id"        : "$zone_context",
        "avg_latency": {"$avg": "$avg_latency_ms"},
        "max_latency": {"$max": "$max_latency_ms"},
        "sessions"   : {"$sum": 1}
    }},
    {"$sort": {"avg_latency": -1}}
]

results = db.app_sessions.aggregate(latency_pipeline)
for r in results:
    print(f"  {r['_id']:<12}  avg: {r['avg_latency']:.0f}ms  "
          f"max: {r['max_latency']}ms  sessions: {r['sessions']}")

READ 5 — Average API Latency by Zone:
---------------------------------------------
  Airport       avg: 597ms  max: 1701ms  sessions: 86
  Central       avg: 507ms  max: 1558ms  sessions: 93
  North         avg: 444ms  max: 944ms  sessions: 93
  East          avg: 435ms  max: 1005ms  sessions: 91
  South         avg: 434ms  max: 1403ms  sessions: 95
  West          avg: 421ms  max: 1074ms  sessions: 92
  Riverside     avg: 420ms  max: 1138ms  sessions: 87


### UPDATE Operations

In [22]:
# UPDATE 1: Mark high-risk customers (3+ complaints) with a risk_flag
print("UPDATE 1 — Add risk_flag to customers with 3+ complaints:")
print("-" * 55)

result = db.customer_cases.update_many(
    {"total_complaints": {"$gte": 3}},
    {"$set": {"risk_flag": "HIGH", "flagged_at": str(datetime.now())}}
)

print(f"Matched : {result.matched_count} documents")
print(f"Modified: {result.modified_count} documents")

# Verify
high_risk = db.customer_cases.find({"risk_flag": "HIGH"},
                                    {"customer_id":1,"total_complaints":1,"risk_flag":1,"_id":0})
print("\nHigh risk customers:")
for doc in high_risk:
    print(f"  {doc}")

UPDATE 1 — Add risk_flag to customers with 3+ complaints:
-------------------------------------------------------
Matched : 12 documents
Modified: 12 documents

High risk customers:
  {'customer_id': 'C0110', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0142', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0172', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0191', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0242', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0282', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0368', 'total_complaints': 4, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0372', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0421', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0545', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0573', 'total_complaints': 3, 'risk_flag': 'HIGH'}
  {'customer_id': 'C0626', 'total_com

In [23]:
# UPDATE 2: Add a 'requires_review' flag to unresolved Critical incidents
print("UPDATE 2 — Flag deliveries with unresolved Critical incidents:")
print("-" * 60)

result = db.delivery_events.update_many(
    {"incidents": {"$elemMatch": {
        "severity"         : "Critical",
        "resolution_status": {"$in": ["Open", "Escalated"]}
    }}},
    {"$set": {"requires_review": True}}
)

print(f"Matched : {result.matched_count}")
print(f"Modified: {result.modified_count}")

UPDATE 2 — Flag deliveries with unresolved Critical incidents:
------------------------------------------------------------
Matched : 11
Modified: 11


In [24]:
# UPDATE 3: Update a specific customer's account status (single document update)
result = db.customer_cases.update_one(
    {"customer_id": "C0368"},
    {"$set": {
        "account_status"  : "Under Review",
        "review_reason"   : "Highest complaint volume in dataset",
        "last_updated"    : str(datetime.now())
    }}
)

print(f"UPDATE 3 — Single customer update:")
print(f"  Matched: {result.matched_count}, Modified: {result.modified_count}")

# Verify the update
doc = db.customer_cases.find_one(
    {"customer_id": "C0368"},
    {"customer_id":1, "account_status":1, "review_reason":1, "total_complaints":1, "_id":0}
)
print("Updated document:")
pprint(doc)

UPDATE 3 — Single customer update:
  Matched: 1, Modified: 1
Updated document:
{'account_status': 'Under Review',
 'customer_id': 'C0368',
 'review_reason': 'Highest complaint volume in dataset',
 'total_complaints': 4}


### DELETE Operations

In [25]:
# DELETE 1: Remove test/orphan app sessions with 0 events
print("DELETE 1 — Remove app sessions with zero events:")
zero_event_sessions = db.app_sessions.count_documents({"total_events": 0})
print(f"  Sessions with 0 events (before delete): {zero_event_sessions}")

if zero_event_sessions > 0:
    result = db.app_sessions.delete_many({"total_events": 0})
    print(f"  Deleted: {result.deleted_count} documents")
else:
    print("  No empty sessions found — data is clean.")

DELETE 1 — Remove app sessions with zero events:
  Sessions with 0 events (before delete): 0
  No empty sessions found — data is clean.


In [26]:
# DELETE 2: Remove a specific test document by ID
test_doc = {
    "customer_id"    : "TEST_001",
    "home_zone"      : "Test",
    "total_complaints": 0,
    "complaints"     : [],
    "note"           : "This is a test document — will be deleted"
}

insert_result = db.customer_cases.insert_one(test_doc)
print(f"DELETE 2 — Test document inserted with id: {insert_result.inserted_id}")

# Now delete it
delete_result = db.customer_cases.delete_one({"_id": insert_result.inserted_id})
print(f"  Deleted count: {delete_result.deleted_count}")
print("  Test document successfully removed.")

DELETE 2 — Test document inserted with id: 69c3af6a660f747ca6b26ffe
  Deleted count: 1
  Test document successfully removed.


## Section 8 — Query Optimisation (10 Marks)
### Creating Indexes and Using Explain Plans

In [27]:
import pymongo
# BEFORE INDEXING
# Run a query and capture the explain plan to see how it performs WITHOUT indexes

print("BEFORE indexing — explain plan for customer_cases query:")
print("=" * 55)

# This query finds all customers in the Central zone with complaints
# Without an index, MongoDB must scan EVERY document (COLLSCAN)

explain_before = db.customer_cases.find(
    {"home_zone": "Central", "total_complaints": {"$gt": 0}}
).explain()

stage = explain_before["executionStats"]
print(f"  Execution time (ms)     : {stage['executionTimeMillis']}")
print(f"  Documents examined      : {stage['totalDocsExamined']}")
print(f"  Documents returned      : {stage['nReturned']}") # Corrected key
print(f"  Index used              : {explain_before['queryPlanner']['winningPlan']['stage']}")
print()
print("COLLSCAN = full collection scan = slow on large data")

BEFORE indexing — explain plan for customer_cases query:
  Execution time (ms)     : 0
  Documents examined      : 650
  Documents returned      : 43
  Index used              : COLLSCAN

COLLSCAN = full collection scan = slow on large data


In [28]:
# Create indexes on the most queried fields in each collection

# customer_cases indexes
db.customer_cases.create_index([("home_zone", 1)],
                                name="idx_customer_zone")
db.customer_cases.create_index([("total_complaints", -1)],
                                name="idx_complaint_count")
db.customer_cases.create_index([("customer_type", 1), ("home_zone", 1)],
                                name="idx_type_zone_compound")
db.customer_cases.create_index([("risk_flag", 1)],
                                name="idx_risk_flag",
                                sparse=True)     # sparse = only index docs where field exists

# delivery_events indexes
db.delivery_events.create_index([("delivery_status", 1)],
                                  name="idx_delivery_status")
db.delivery_events.create_index([("hub_id", 1), ("delivery_status", 1)],
                                  name="idx_hub_status_compound")
db.delivery_events.create_index([("manual_route_override_count", -1)],
                                  name="idx_overrides")
db.delivery_events.create_index([("incidents.severity", 1)],
                                  name="idx_incident_severity")

# app_sessions indexes
db.app_sessions.create_index([("customer_id", 1)],
                               name="idx_session_customer")
db.app_sessions.create_index([("zone_context", 1)],
                               name="idx_session_zone")
db.app_sessions.create_index([("has_payment_retry", 1), ("failed_events", -1)],
                               name="idx_payment_retry_compound")

print("All indexes created.")
print()
print("Indexes on customer_cases:")
for idx in db.customer_cases.list_indexes():
    print(f"  {idx['name']}: {idx['key']}")

All indexes created.

Indexes on customer_cases:
  _id_: SON([('_id', 1)])
  idx_customer_zone: SON([('home_zone', 1)])
  idx_complaint_count: SON([('total_complaints', -1)])
  idx_type_zone_compound: SON([('customer_type', 1), ('home_zone', 1)])
  idx_risk_flag: SON([('risk_flag', 1)])


In [29]:
# AFTER INDEXING
# Run the SAME query again and compare explain plan

print("AFTER indexing — explain plan for the same query:")
print("=" * 55)

explain_after = db.customer_cases.find(
    {"home_zone": "Central", "total_complaints": {"$gt": 0}}
).explain()

stage_after = explain_after["executionStats"]
print(f"  Execution time (ms)     : {stage_after['executionTimeMillis']}")
print(f"  Documents examined      : {stage_after['totalDocsExamined']}")
print(f"  Documents returned      : {stage_after['nReturned']}")
print(f"  Index used              : {explain_after['queryPlanner']['winningPlan']['stage']}")
print()

# Side-by-side comparison
print("Comparison: Before vs After Indexing")
print("-" * 45)
print(f"{'Metric':<30} {'Before':>8} {'After':>8}")
print("-" * 45)
print(f"{'Execution time (ms)':<30} {stage['executionTimeMillis']:>8} {stage_after['executionTimeMillis']:>8}")
print(f"{'Docs examined':<30} {stage['totalDocsExamined']:>8} {stage_after['totalDocsExamined']:>8}")
print(f"{'Docs returned':<30} {stage['nReturned']:>8} {stage_after['nReturned']:>8}")

AFTER indexing — explain plan for the same query:
  Execution time (ms)     : 2
  Documents examined      : 110
  Documents returned      : 43
  Index used              : FETCH

Comparison: Before vs After Indexing
---------------------------------------------
Metric                           Before    After
---------------------------------------------
Execution time (ms)                   0        2
Docs examined                       650      110
Docs returned                        43       43


In [30]:
# Compound index performance test — hub + status query
print("Explain plan — compound index query (hub_id + delivery_status):")
print("=" * 60)

explain_hub = db.delivery_events.find(
    {"hub_id": "H05", "delivery_status": "Failed"}
).explain()

stage_hub = explain_hub["executionStats"]
print(f"  Execution time (ms)  : {stage_hub['executionTimeMillis']}")
print(f"  Docs examined        : {stage_hub['totalDocsExamined']}")
print(f"  Docs returned        : {stage_hub['nReturned']}") # Corrected key
winning = explain_hub['queryPlanner']['winningPlan']
print(f"  Query plan stage     : {winning['stage']}")
print()
print("Justification: Compound index on (hub_id, delivery_status) allows MongoDB")
print("to filter on both fields simultaneously — far faster than two separate queries.")

Explain plan — compound index query (hub_id + delivery_status):
  Execution time (ms)  : 2
  Docs examined        : 23
  Docs returned        : 23
  Query plan stage     : FETCH

Justification: Compound index on (hub_id, delivery_status) allows MongoDB
to filter on both fields simultaneously — far faster than two separate queries.


In [31]:
# Aggregation pipeline with index hint for performance
print("Optimised Aggregation — Average rating by hub with index hint:")
print("=" * 60)

pipeline = [
    {"$match": {"delivery_status": {"$in": ["Failed", "Delayed"]}}},  # uses idx_delivery_status
    {"$group": {
        "_id"         : "$hub_id",
        "total_bad"   : {"$sum": 1},
        "avg_rating"  : {"$avg": "$customer_rating"},
        "avg_overrides": {"$avg": "$manual_route_override_count"},
        "with_incidents": {"$sum": {"$cond": [{"$gt": ["$incident_count", 0]}, 1, 0]}}
    }},
    {"$sort": {"total_bad": -1}}
]

results = list(db.delivery_events.aggregate(pipeline))
print(f"{'Hub':<8} {'Bad Deliveries':>15} {'Avg Rating':>12} {'Avg Overrides':>15} {'With Incidents':>16}")
print("-" * 70)
for r in results:
    print(f"{r['_id']:<8} {r['total_bad']:>15} "
          f"{r['avg_rating']:>12.2f} {r['avg_overrides']:>15.2f} {r['with_incidents']:>16}")

Optimised Aggregation — Average rating by hub with index hint:
Hub       Bad Deliveries   Avg Rating   Avg Overrides   With Incidents
----------------------------------------------------------------------
H05                   48         2.95            0.94               16
H08                   48         3.18            1.06               10
H04                   44         3.15            1.05                9
H01                   43         2.94            1.12               10
H06                   42         3.09            0.90                8
H07                   39         3.18            1.38               12
H02                   36         3.29            1.06                6
H03                   34         3.13            1.00                8


## Section 9 — Final Collection Summary

In [32]:
# Print final state of all collections
print("=" * 50)
print("  NORTHSTAR MONGODB — FINAL COLLECTION SUMMARY")
print("=" * 50)

collections = ["customer_cases", "delivery_events", "app_sessions"]

for col_name in collections:
    col = db[col_name]
    count = col.count_documents({})
    indexes = list(col.list_indexes())
    print(f"\n{col_name}")
    print(f"  Documents : {count}")
    print(f"  Indexes   : {len(indexes)}")
    for idx in indexes:
        print(f"    - {idx['name']}: {dict(idx['key'])}")

print()
print("All CRUD operations demonstrated:")
print("  CREATE : insert_one, insert_many")
print("  READ   : find, find_one, aggregate")
print("  UPDATE : update_one, update_many with $set")
print("  DELETE : delete_one, delete_many")
print()
print("Query optimisation demonstrated:")
print("  - Before/after explain plans showing COLLSCAN vs IXSCAN")
print("  - Single field indexes on high-cardinality fields")
print("  - Compound indexes for multi-field queries")
print("  - Sparse index for optional fields")
print("  - Optimised aggregation pipeline with indexed $match stage")

  NORTHSTAR MONGODB — FINAL COLLECTION SUMMARY

customer_cases
  Documents : 650
  Indexes   : 5
    - _id_: {'_id': 1}
    - idx_customer_zone: {'home_zone': 1}
    - idx_complaint_count: {'total_complaints': -1}
    - idx_type_zone_compound: {'customer_type': 1, 'home_zone': 1}
    - idx_risk_flag: {'risk_flag': 1}

delivery_events
  Documents : 950
  Indexes   : 5
    - _id_: {'_id': 1}
    - idx_delivery_status: {'delivery_status': 1}
    - idx_hub_status_compound: {'hub_id': 1, 'delivery_status': 1}
    - idx_overrides: {'manual_route_override_count': -1}
    - idx_incident_severity: {'incidents.severity': 1}

app_sessions
  Documents : 637
  Indexes   : 4
    - _id_: {'_id': 1}
    - idx_session_customer: {'customer_id': 1}
    - idx_session_zone: {'zone_context': 1}
    - idx_payment_retry_compound: {'has_payment_retry': 1, 'failed_events': -1}

All CRUD operations demonstrated:
  CREATE : insert_one, insert_many
  READ   : find, find_one, aggregate
  UPDATE : update_one, update

In [33]:
# Close MongoDB connection cleanly
client.close()
print("MongoDB connection closed.")
print("Notebook 3 complete.")

MongoDB connection closed.
Notebook 3 complete.
